In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("gold_layer").getOrCreate()

In [0]:
df_silver = spark.table("teleco_churn_datalakehouse.silver.teleco_customer_churn")

In [0]:
df_silver.display()

- Creating the dimesnion tables

In [0]:
from pyspark.sql import functions as F

- Creating the dimension tables

In [0]:
# Creating the dim_customer dimension table.
dim_customer = df_silver.select(
    "CustomerID",
    "Gender",
    "Senior_Citizen",
    "Partner",
    "Dependents"
).dropDuplicates()

# Add surrogate key
from pyspark.sql.window import Window

window_spec = Window.orderBy("CustomerID")

dim_customer = dim_customer.withColumn(
    "customer_key",
    F.row_number().over(window_spec)
)

In [0]:
# Creating the dim_plan table
dim_plan = df_silver.select(
    "Contract",
    "Internet_Service",
    "Payment_Method"
).dropDuplicates()

# Define window partitioned by unique columns to avoid performance degradation
window_spec = Window.orderBy("Contract")

dim_plan = dim_plan.withColumn(
    "plan_key",
    F.row_number().over(window_spec)
)




- Creating the fact table

In [0]:
# To avoid performance degradation, define a partition in the window specification for dim_customer
window_spec = Window.partitionBy("CustomerID").orderBy("CustomerID")

dim_customer = dim_customer.withColumn(
    "customer_key",
    F.row_number().over(window_spec)
)

df_silver_alias = df_silver.alias("s")
dim_customer_alias = dim_customer.alias("c")
dim_plan_alias = dim_plan.alias("p")

fact_table = df_silver_alias \
    .join(dim_customer_alias, F.col("s.CustomerID") == F.col("c.CustomerID")) \
    .join(
        dim_plan_alias,
        (F.col("s.Contract") == F.col("p.Contract")) &
        (F.col("s.Internet_Service") == F.col("p.Internet_Service")) &
        (F.col("s.Payment_Method") == F.col("p.Payment_Method"))
    )

- Adding buisness metrics

In [0]:
# Churn flag business metric
fact_table = fact_table.withColumn(
    "churn_flag",
    F.col("s.Churn_value").cast("int")
)

# revenue at risk
fact_table = fact_table.withColumn(
    "revenue_at_risk",
    F.when(F.col("churn_flag") == 1, F.col("s.Monthly_Charges")).otherwise(0)
)

In [0]:
# Selecting only the fact columns
fact_customer_subscription = fact_table.select(
    "customer_key",
    "plan_key",
    "Tenure_Months",
    "Monthly_Charges",
    "Total_Charges",
    "churn_flag",
    "revenue_at_risk"
)

- Writing the dimension and fact tables to the gold schema

In [0]:
# Writing dimension tables
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.gold.dim_customer")

dim_plan.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.gold.dim_plan")

In [0]:
# Writing fact table
fact_customer_subscription.write \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.gold.fact_customer_subscription")

- Working on the churn rate KPI and saving the table

In [0]:
kpi_churn = fact_customer_subscription.agg(
    F.count("*").alias("total_customers"),
    F.sum("churn_flag").alias("churned_customers"),
    F.sum("revenue_at_risk").alias("total_revenue_at_risk")
).withColumn(
    "churn_rate",
    F.col("churned_customers") / F.col("total_customers")
)

In [0]:
kpi_churn.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.gold.kpi_churn")

- Confirm data entry into the four delta tables created

In [0]:
%sql
-- Checking the data entered ubto dim_customer delta table
SELECT * 
FROM teleco_churn_datalakehouse.gold.dim_customer

In [0]:
%sql
-- Checking the data entered ubto dim_plan delta table
SELECT * 
FROM teleco_churn_datalakehouse.gold.dim_plan

In [0]:
%sql
-- Checking the data entered ubto fact_customer_subscription delta table
SELECT * 
FROM teleco_churn_datalakehouse.gold.fact_customer_subscription

In [0]:
%sql
-- Checking the data entered ubto kpi_churn delta table
SELECT * 
FROM teleco_churn_datalakehouse.gold.kpi_churn